# Visualization
Using the .trace option on cameras and telescopes allows for the visualization of the trajectory of the rays. For this, they have a `record_trajectory` option, which causes the `.trajectory` object of the NamedTuple `TraceResult` to be filled. For this to work we need to a create actual rays to pass to the telescope:

In [1]:
import jax
import jax.numpy as jnp

from iactrace import Camera, Telescope, show_camera, show_telescope

In [2]:
n_rays = 100
key1, key2 = jax.random.split(jax.random.key(123))

r = 6.0 * jnp.sqrt(jax.random.uniform(key1, (n_rays,)))
theta = jax.random.uniform(key2, (n_rays,)) * 2 * jnp.pi
ray_origins = jnp.stack(
    [r * jnp.cos(theta), r * jnp.sin(theta), jnp.ones(n_rays) * 20.0],
    axis=1,
)

tilt_angle = 0 * jnp.pi / 180
ray_directions = jnp.broadcast_to(
    jnp.array([jnp.sin(tilt_angle), 0.0, -jnp.cos(tilt_angle)]),
    (n_rays, 3),
)
ray_values = jnp.ones(n_rays)

We import HESS CT3 as our example of choice:

In [3]:
telescope = Telescope.from_yaml(
    "../../configs/HESS/CT3.yaml",
    n_samples=128,
    key=jax.random.key(42),
)
camera = Camera.from_yaml("../../configs/HESS/HESS1U.yaml")

## Optics Path
Tracing the rays, we need to set `record_trajectory = True`:

In [4]:
rays, traj = telescope.trace(ray_origins, ray_directions, ray_values, record_trajectory=True)

We can then use the viz of the telescope:

In [5]:
scene = show_telescope(telescope, trajectory=traj, camera=camera)
scene.show(viewer="jupyter")

## Camera Path
For the camera, we can do the same thing via camera.trace:

In [6]:
c_rays, c_traj = camera.trace(rays)
scene = show_camera(camera, trajectory=c_traj)
scene.show(viewer="jupyter")